<a href="https://colab.research.google.com/github/bhar-gav/machine_learning/blob/main/A4_dropout_exp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/wandb/examples/blob/master/colabs/pytorch/Organizing_Hyperparameter_Sweeps_in_PyTorch_with_W&B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<!--- @wandbcode{sweeps-video} -->

In [1]:
!pip install wandb -Uq

2. Import W&B:

In [2]:
import wandb

3. Log in to W&B and provide your API key when prompted:

In [3]:
wandb.login()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bhar_gav (bhar_gav-national-institute-of-technology-hamirpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Pick a search method

First, specify a hyperparameter search method within your configuration dictionary. [There are three hyperparameter search strategies to choose from: grid, random, and Bayesian search](https://docs.wandb.ai/guides/sweeps/sweep-config-keys#method).

For this tutorial, you will use a random search. Within your notebook, create a dictionary and specify `random` for the `method` key.

In [4]:
sweep_config = {
    'method': 'random'
    }

Specify a metric that you want to optimize for. You do not need to specify the metric and goal for sweeps that use random search method. However, it is good practice to keep track of your sweep goals because you can refer to it at a later time.

In [5]:
metric = {
    'name': 'validation_accuracy',
    'goal': 'maximize'
    }

sweep_config['metric'] = metric

In [17]:
parameters_dict= {
        'epochs': {'values': [10]},
        'lr': {'values': [0.001, 0.01]},
        'momentum': {'values': [0.9, 0.99]},
        'optimizer': {'values': ['sgd']},
        'batch_size': {'values': [64]},
        'weight_init': {'values': ['random']},
        'dropout_prob': {'values': [0.2, 0.3, 0.5]},  # Dropout probability between 20% to 50%
        'dropout_method': {'values': ['random', 'dropconnect', 'dropblock', 'maxdropout', 'biased_dropout', 'flipover']},
        'model': {'values': ['create_standard_network_1', 'create_standard_network_2', 'create_dropout_network_logistic', 'create_dropout_network_relu']}
    }

sweep_config['parameters'] = parameters_dict

In [18]:
import pprint
pprint.pprint(sweep_config)

{'method': 'random',
 'metric': {'goal': 'maximize', 'name': 'validation_accuracy'},
 'parameters': {'batch_size': {'values': [64]},
                'dropout_method': {'values': ['random',
                                              'dropconnect',
                                              'dropblock',
                                              'maxdropout',
                                              'biased_dropout',
                                              'flipover']},
                'dropout_prob': {'values': [0.2, 0.3, 0.5]},
                'epochs': {'values': [10]},
                'lr': {'values': [0.001, 0.01]},
                'model': {'values': ['create_standard_network_1',
                                     'create_standard_network_2',
                                     'create_dropout_network_logistic',
                                     'create_dropout_network_relu']},
                'momentum': {'values': [0.9, 0.99]},
                'optimizer

## Step 2️: Initialize the Sweep

\

In [19]:
sweep_id = wandb.sweep(sweep_config, project="nn_dropout")

Create sweep with ID: cmd9ecou
Sweep URL: https://wandb.ai/bhar_gav-national-institute-of-technology-hamirpur/nn_dropout/sweeps/cmd9ecou


## Step 3:  Define your deep learning code



In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# Initialize device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset building function
def build_dataset(batch_size):
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])  # Normalize for MNIST
    dataset = datasets.MNIST('.', train=True, download=True, transform=transform)
    # Split 10% for validation
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader


# Neural Network Architecture with Dropout
class NeuralNetworkWithDropout(nn.Module):
    def __init__(self, input_size, hidden_layers, output_size, dropout_prob, activation_fn, init_method, max_threshold=None):
        super(NeuralNetworkWithDropout, self).__init__()
        self.init_method = init_method
        self.max_threshold = max_threshold

        layers = []

        # Input to first hidden layer
        layers.append(nn.Linear(input_size, hidden_layers[0]))
        layers.append(activation_fn())
        layers.append(nn.Dropout(dropout_prob))

        # Hidden layers with dropout
        for i in range(1, len(hidden_layers)):
            layers.append(nn.Linear(hidden_layers[i-1], hidden_layers[i]))
            layers.append(activation_fn())
            layers.append(nn.Dropout(dropout_prob))

        # Output layer
        layers.append(nn.Linear(hidden_layers[-1], output_size))

        self.network = nn.Sequential(*layers)
        self.apply(self._initialize_weights)

    def forward(self, x):
      # Flatten the input tensor
      x = x.view(x.size(0), -1)  # Flatten the 28x28 images into a 784 vector
      return self.network(x)


    def _initialize_weights(self, layer):
        if isinstance(layer, nn.Linear):
            if self.init_method == 'random':
                nn.init.normal_(layer.weight, mean=0, std=0.01)
            elif self.init_method == 'max_threshold':
                nn.init.normal_(layer.weight, mean=0, std=0.01)
                if self.max_threshold:
                    torch.clamp(layer.weight, max=self.max_threshold)
            elif self.init_method == 'pretraining':
                nn.init.normal_(layer.weight, mean=0, std=0.01)

            if layer.bias is not None:
                nn.init.constant_(layer.bias, 0)


# Experiment Configurations

# 1. StandardNeuralNet Logistic 2 layers, 100 units
def create_standard_network_1():
    return NeuralNetworkWithDropout(input_size=784, hidden_layers=[100, 100], output_size=10,
                                    dropout_prob=0.5, activation_fn=nn.Sigmoid, init_method='random')

# 2. StandardNeuralNet Logistic 2 layers, 800 units
def create_standard_network_2():
    return NeuralNetworkWithDropout(input_size=784, hidden_layers=[800, 800], output_size=10,
                                    dropout_prob=0.5, activation_fn=nn.Sigmoid, init_method='random')

# 3. DropoutNN Logistic 3 layers, 1024 units
def create_dropout_network_logistic():
    return NeuralNetworkWithDropout(input_size=784, hidden_layers=[1024, 1024, 1024], output_size=10,
                                    dropout_prob=0.5, activation_fn=nn.Sigmoid, init_method='random')

# 4. DropoutNN ReLU 3 layers, 1024 units
def create_dropout_network_relu():
    return NeuralNetworkWithDropout(input_size=784, hidden_layers=[1024, 1024, 1024], output_size=10,
                                    dropout_prob=0.5, activation_fn=nn.ReLU, init_method='random')


# DROPOUTS

# Define custom DropConnect layer
class DropConnect(nn.Module):
    def __init__(self, layer, p=0.5):
        super(DropConnect, self).__init__()
        self.layer = layer
        self.p = p

    def forward(self, x):
        if self.training:
            # DropConnect: Randomly zero out weights, not activations
            mask = (torch.rand_like(self.layer.weight) > self.p).float()
            weight = self.layer.weight * mask
            return F.linear(x, weight, self.layer.bias)
        else:
            return self.layer(x)

# Define custom DropBlock layer
class DropBlock(nn.Module):
    def __init__(self, p=0.5):
        super(DropBlock, self).__init__()
        self.p = p

    def forward(self, x):
        if self.training:
            # DropBlock: Randomly block entire blocks of activations
            block_size = int(x.size(1) * self.p)
            mask = torch.ones_like(x)
            mask[:, :block_size] = 0  # You can modify this logic to randomly block in more advanced ways
            x = x * mask
        return x

# Define Maxdropout (drop the largest activations)
class MaxDropout(nn.Module):
    def __init__(self, p=0.5):
        super(MaxDropout, self).__init__()
        self.p = p

    def forward(self, x):
        if self.training:
            # Drop the max activations
            top_k = int(x.size(1) * self.p)
            _, indices = torch.topk(x, top_k, dim=1, largest=True, sorted=False)
            mask = torch.zeros_like(x)
            mask.scatter_(1, indices, 1)
            x = x * mask
        return x

# Define Biased Dropout
class BiasedDropout(nn.Module):
    def __init__(self, p=0.5, bias=0.2):
        super(BiasedDropout, self).__init__()
        self.p = p
        self.bias = bias

    def forward(self, x):
        if self.training:
            # Biased Dropout: Apply biased dropout, where some neurons are more likely to be dropped
            prob = torch.full_like(x, self.p + self.bias)
            mask = (torch.rand_like(x) > prob).float()
            x = x * mask
        return x

# Define Flipover Dropout
class FlipoverDropout(nn.Module):
    def __init__(self, p=0.5):
        super(FlipoverDropout, self).__init__()
        self.p = p

    def forward(self, x):
        if self.training:
            # Flipover: Randomly negate the activations of dropped units
            mask = (torch.rand_like(x) > self.p).float()
            x = x * mask
            x = x - (x * mask)  # Negate the dropped values
        return x

# Main function to apply different dropout methods
def apply_dropout_method(model, method_name, dropout_prob=0.5):
    if method_name == "random":
        # Apply standard random dropout to each layer
        for module in model.children():
            if isinstance(module, nn.Linear):
                module.dropout = nn.Dropout(dropout_prob)
        return model

    if method_name == "dropconnect":
        # Apply DropConnect
        for module in model.children():
            if isinstance(module, nn.Linear):
                module = DropConnect(module, p=dropout_prob)
        return model

    if method_name == "dropblock":
        # Apply DropBlock
        for module in model.children():
            if isinstance(module, nn.Linear):
                module = DropBlock(p=dropout_prob)
        return model

    if method_name == "maxdropout":
        # Apply Maxdropout
        for module in model.children():
            if isinstance(module, nn.Linear):
                module = MaxDropout(p=dropout_prob)
        return model

    if method_name == "biased_dropout":
        # Apply Biased Dropout
        for module in model.children():
            if isinstance(module, nn.Linear):
                module = BiasedDropout(p=dropout_prob)
        return model

    if method_name == "flipover":
        # Apply Flipover Dropout
        for module in model.children():
            if isinstance(module, nn.Linear):
                module = FlipoverDropout(p=dropout_prob)
        return model

    # Default: no dropout
    return model


# Optimizer function
def get_optimizer(model, optimizer_name, lr, momentum=0, weight_decay=0):
    if optimizer_name == 'sgd':
        return optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    else:
        raise ValueError("Optimizer not supported")


# Training function
def train(model, train_loader, optimizer, criterion, epochs):
        config = wandb.config

        model.train()
        for epoch in range(epochs):
            running_loss = 0.0
            correct = 0
            total = 0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                # Forward pass
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

            wandb.log({
                "epoch": epoch + 1,
                "train_loss": running_loss / len(train_loader),
                "train_accuracy": 100 * correct / total,
                "trial_name": f"m_{config.model}_dr_{config.dropout_method}_p_{config.dropout_prob}lr_{config.lr}_m_{config.momentum}"  # Add trial name
            })

            print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader)}, Accuracy: {100 * correct / total}%")

# Evaluation function
def evaluate(model, val_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy




In [21]:
import wandb

def run_experiment():
    # Initialize a new wandb run
    with wandb.init() as run:
        # Access the sweep config from wandb
        config = wandb.config

        # Generate a custom trial name using hyperparameters from the config
        trial_name = f"m_{config.model}_dr_{config.dropout_method}_p_{config.dropout_prob}lr_{config.lr}_m_{config.momentum}"

        run.name = trial_name

        # Build dataset for training and validation
        train_loader, val_loader = build_dataset(config.batch_size)

        # Choose model based on config
        if config.model == 'create_standard_network_1':
            model = create_standard_network_1().to(device)
        elif config.model == 'create_standard_network_2':
            model = create_standard_network_2().to(device)
        elif config.model == 'create_dropout_network_logistic':
            model = create_dropout_network_logistic().to(device)
        elif config.model == 'create_dropout_network_relu':
            model = create_dropout_network_relu().to(device)
        else:
            raise ValueError(f"Unknown model: {config.model}")

        # Apply the selected dropout method
        model = apply_dropout_method(model, config.dropout_method, dropout_prob=config.dropout_prob)

        # Define loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = get_optimizer(model, config.optimizer, config.lr)

        # Train the model
        train(model, train_loader, optimizer, criterion, config.epochs)

        # Evaluate the model
        val_accuracy = evaluate(model, val_loader)
        wandb.log({"validation_accuracy": val_accuracy, "trial_name": trial_name})




In [ ]:
# Run the sweep
wandb.agent(sweep_id, run_experiment,count=15)

wandb: Agent Starting Run: hxe40ux0 with config:
wandb: 	batch_size: 64
wandb: 	dropout_method: flipover
wandb: 	dropout_prob: 0.3
wandb: 	epochs: 10
wandb: 	lr: 0.001
wandb: 	model: create_dropout_network_logistic
wandb: 	momentum: 0.99
wandb: 	optimizer: sgd
wandb: 	weight_init: random


Epoch 1, Loss: 2.313715978256334, Accuracy: 10.344444444444445%
Epoch 2, Loss: 2.313852319220231, Accuracy: 10.264814814814814%
Epoch 3, Loss: 2.3131088258530856, Accuracy: 10.266666666666667%
Epoch 4, Loss: 2.31235595619509, Accuracy: 10.255555555555556%
Epoch 5, Loss: 2.311236272490985, Accuracy: 10.64074074074074%
Epoch 6, Loss: 2.3116681776340537, Accuracy: 10.424074074074074%
Epoch 7, Loss: 2.3126468520028896, Accuracy: 10.12962962962963%
Epoch 8, Loss: 2.3098923880342057, Accuracy: 10.592592592592593%
Epoch 9, Loss: 2.310816833758241, Accuracy: 10.42962962962963%
Epoch 10, Loss: 2.309013242687659, Accuracy: 10.39074074074074%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▄▃▃▃█▅▁▇▅▅
train_loss,██▇▆▄▅▆▂▄▁
validation_accuracy,▁
epoch,10
train_accuracy,10.39074
train_loss,2.30901
trial_name,m_create_dropout_net...
validation_accuracy,11.5


wandb: Agent Starting Run: ow1qwsaw with config:
wandb: 	batch_size: 64
wandb: 	dropout_method: flipover
wandb: 	dropout_prob: 0.3
wandb: 	epochs: 10
wandb: 	lr: 0.01
wandb: 	model: create_dropout_network_relu
wandb: 	momentum: 0.99
wandb: 	optimizer: sgd
wandb: 	weight_init: random


Epoch 1, Loss: 2.282676828981011, Accuracy: 26.738888888888887%
Epoch 2, Loss: 1.3389937177441695, Accuracy: 59.02777777777778%
Epoch 3, Loss: 0.5518980857814658, Accuracy: 83.1462962962963%
Epoch 4, Loss: 0.3990687712038298, Accuracy: 88.1462962962963%
Epoch 5, Loss: 0.31922600320341743, Accuracy: 90.43703703703704%
Epoch 6, Loss: 0.2669146334381677, Accuracy: 92.11111111111111%
Epoch 7, Loss: 0.2275311391650571, Accuracy: 93.28333333333333%
Epoch 8, Loss: 0.19736669842436275, Accuracy: 94.25%
Epoch 9, Loss: 0.1752340656780236, Accuracy: 94.80555555555556%
Epoch 10, Loss: 0.15861722357544641, Accuracy: 95.35%


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▇▇▇█████
train_loss,█▅▂▂▂▁▁▁▁▁
validation_accuracy,▁
epoch,10
train_accuracy,95.35
train_loss,0.15862
trial_name,m_create_dropout_net...
validation_accuracy,95.66667


wandb: Agent Starting Run: w8vzanmo with config:
wandb: 	batch_size: 64
wandb: 	dropout_method: biased_dropout
wandb: 	dropout_prob: 0.2
wandb: 	epochs: 10
wandb: 	lr: 0.001
wandb: 	model: create_standard_network_2
wandb: 	momentum: 0.99
wandb: 	optimizer: sgd
wandb: 	weight_init: random


Epoch 1, Loss: 2.3102281909983304, Accuracy: 10.274074074074074%
Epoch 2, Loss: 2.309920120860728, Accuracy: 10.222222222222221%
Epoch 3, Loss: 2.3090527162167698, Accuracy: 10.672222222222222%
Epoch 4, Loss: 2.307859050154121, Accuracy: 10.614814814814816%
Epoch 5, Loss: 2.308108908587722, Accuracy: 10.7%
Epoch 6, Loss: 2.306145212379112, Accuracy: 10.733333333333333%




---



end

## Visualize Sweep Results


## Learn more about W&B Sweeps

We created a simple training script and [a few flavors of sweep configs](https://github.com/wandb/examples/tree/master/examples/keras/keras-cnn-fashion) for you to play with. We highly encourage you to give these a try.

That repo also has examples to help you try more advanced sweep features like [Bayesian Hyperband](https://app.wandb.ai/wandb/examples-keras-cnn-fashion/sweeps/us0ifmrf?workspace=user-lavanyashukla), and [Hyperopt](https://app.wandb.ai/wandb/examples-keras-cnn-fashion/sweeps/xbs2wm5e?workspace=user-lavanyashukla).